# Bronze Ingestion

This notebook incrementally ingests transaction and route files from
Amazon S3 using Databricks Auto Loader.

The pipeline writes source records to Bronze Delta tables while preserving
file-level lineage, ingestion timestamps, rescued data, and execution
metadata.

## Responsibilities

- Read source files from Unity Catalog external volumes
- Apply explicit source schemas
- Process only newly arrived files
- Preserve source and ingestion metadata
- Capture malformed records for downstream validation
- Write append-only Bronze Delta tables
- Maintain independent checkpoints for each ingestion stream

In [0]:
import uuid

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    DecimalType,
    TimestampType,
    IntegerType,
)

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("platform_schema", "sbx_platform")
dbutils.widgets.text("bronze_schema", "sbx_bronze")
dbutils.widgets.text("run_id", "")

In [0]:
catalog = dbutils.widgets.get("catalog").strip()
platform_schema = dbutils.widgets.get("platform_schema").strip()
bronze_schema = dbutils.widgets.get("bronze_schema").strip()
provided_run_id = dbutils.widgets.get("run_id").strip()

required_parameters = {
    "catalog": catalog,
    "platform_schema": platform_schema,
    "bronze_schema": bronze_schema,
}

missing_parameters = [
    name
    for name, value in required_parameters.items()
    if not value
]

if missing_parameters:
    raise ValueError(
        f"Missing required parameters: {', '.join(missing_parameters)}"
    )

run_id = provided_run_id or str(uuid.uuid4())

print("Runtime configuration loaded successfully.")
print(f"Catalog: {catalog}")
print(f"Platform schema: {platform_schema}")
print(f"Bronze schema: {bronze_schema}")
print(f"Run ID: {run_id}")

In [0]:
volume_root = f"/Volumes/{catalog}/{platform_schema}"

transactions_source_path = (
    f"{volume_root}/landing/transactions"
)
routes_source_path = (
    f"{volume_root}/landing/routes"
)

transactions_checkpoint_path = (
    f"{volume_root}/checkpoints/bronze_transactions"
)
routes_checkpoint_path = (
    f"{volume_root}/checkpoints/bronze_routes"
)

bronze_transactions_table = (
    f"{catalog}.{bronze_schema}.transactions"
)
bronze_routes_table = (
    f"{catalog}.{bronze_schema}.routes"
)

print("Pipeline paths configured successfully.")
print(f"Transactions source: {transactions_source_path}")
print(f"Routes source: {routes_source_path}")
print(f"Transactions checkpoint: {transactions_checkpoint_path}")
print(f"Routes checkpoint: {routes_checkpoint_path}")
print(f"Transactions target: {bronze_transactions_table}")
print(f"Routes target: {bronze_routes_table}")

In [0]:
def list_non_empty_csv_files(source_path: str):
    try:
        source_objects = dbutils.fs.ls(source_path)
    except Exception as exc:
        raise RuntimeError(
            f"Unable to access source path: {source_path}"
        ) from exc

    csv_files = [
        source_object
        for source_object in source_objects
        if source_object.name.lower().endswith(".csv")
        and source_object.size > 0
    ]

    if not csv_files:
        raise FileNotFoundError(
            f"No non-empty CSV files found at: {source_path}"
        )

    return csv_files


transactions_source_files = list_non_empty_csv_files(
    transactions_source_path
)

routes_source_files = list_non_empty_csv_files(
    routes_source_path
)

print(
    f"Transactions source files found: "
    f"{len(transactions_source_files)}"
)
print(
    f"Routes source files found: "
    f"{len(routes_source_files)}"
)

display(transactions_source_files)
display(routes_source_files)

In [0]:
transactions_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("booking_date", DateType(), True),
    StructField("route_code", StringType(), True),
    StructField("amount", DecimalType(12, 2), True),
    StructField("currency", StringType(), True),
    StructField("status", StringType(), True),
    StructField("updated_at", TimestampType(), True),
])

routes_schema = StructType([
    StructField("route_code", StringType(), True),
    StructField("origin", StringType(), True),
    StructField("destination", StringType(), True),
    StructField("distance_km", IntegerType(), True),
    StructField("region", StringType(), True),
])

print("Source schemas defined successfully.")

### Notes: Auto Loader

- Auto Loader incrementally discovers and processes files arriving in cloud
  object storage.

- It uses the Spark Structured Streaming API, but it can also run as a
  bounded incremental batch using an `availableNow` trigger.

- An explicit schema acts as a source data contract and prevents unreliable
  type inference.

- Rescue mode preserves unexpected columns and values that cannot be parsed
  into the expected data types. These values are captured in
  `_rescued_data` instead of immediately failing the entire ingestion.

- Rescue mode handles technical schema mismatches. It does not replace
  business data-quality validation, such as checking for missing IDs,
  invalid statuses, or negative transaction amounts.

- Source metadata provides record-level lineage:
  - `_source_file` identifies the originating S3 object.
  - `_source_modified_at` captures the source object's modification time.
  - `_ingested_at` records when Databricks processed the record.
  - `_ingestion_run_id` links the record to a specific pipeline execution.

- Defining a streaming DataFrame does not immediately process data because
  Spark uses lazy evaluation. Processing begins only when a streaming write
  is started.

- A persistent and query-specific checkpoint is required to track ingestion
  progress and avoid reprocessing previously handled files.

- Auto Loader, checkpointing, and a Delta Lake target provide exactly-once
  file-processing semantics. Downstream transformations must still be
  designed to be idempotent.

In [0]:

transactions_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("header", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .option("dateFormat", "yyyy-MM-dd")
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .schema(transactions_schema)
    .load(transactions_source_path)
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path"),
    )
    .withColumn(
        "_source_modified_at",
        F.col("_metadata.file_modification_time"),
    )
    .withColumn(
        "_ingested_at",
        F.current_timestamp(),
    )
    .withColumn(
        "_ingestion_run_id",
        F.lit(run_id),
    )
)

transactions_stream_df.printSchema()



In [0]:
transactions_query = (
    transactions_stream_df.writeStream
    .queryName("bronze_transactions_ingestion")
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        transactions_checkpoint_path,
    )
    .trigger(availableNow=True) #Process all currently available unprocessed files and then stop.
    .toTable(bronze_transactions_table)
)

transactions_query.awaitTermination()

print("Transaction ingestion completed successfully.")
print(f"Target table: {bronze_transactions_table}")
print(f"Checkpoint: {transactions_checkpoint_path}")
print(f"Run ID: {run_id}")

In [0]:
transactions_bronze_df = spark.table(
    bronze_transactions_table
)

transaction_metrics = (
    transactions_bronze_df
    .agg(
        F.count("*").alias("total_table_records"),
        F.sum(
            F.when(
                F.col("_ingestion_run_id") == run_id,
                1,
            ).otherwise(0)
        ).alias("current_run_records"),
        F.countDistinct(
            F.when(
                F.col("_ingestion_run_id") == run_id,
                F.col("_source_file"),
            )
        ).alias("current_run_source_files"),
        F.sum(
            F.when(
                (F.col("_ingestion_run_id") == run_id)
                & F.col("_rescued_data").isNotNull(),
                1,
            ).otherwise(0)
        ).alias("current_run_rescued_records"),
    )
    .first()
)

total_table_records = transaction_metrics["total_table_records"]
current_run_records = transaction_metrics["current_run_records"] or 0
current_run_source_files = (
    transaction_metrics["current_run_source_files"] or 0
)
current_run_rescued_records = (
    transaction_metrics["current_run_rescued_records"] or 0
)

print("Transaction Bronze ingestion metrics")
print(f"Total table records: {total_table_records}")
print(f"Current run records: {current_run_records}")
print(f"Current run source files: {current_run_source_files}")
print(
    f"Current run rescued records: "
    f"{current_run_rescued_records}"
)

if current_run_records == 0:
    print(
        "No new transaction records were processed in this run. "
        "This is expected when all source files are already tracked "
        "by the checkpoint."
    )

if current_run_rescued_records > 0:
    print(
        "Warning: rescued transaction records require "
        "data-quality review."
    )

In [0]:
display(
    transactions_bronze_df
    .orderBy(
        F.col("_ingested_at").desc(),
        F.col("transaction_id"),
    )
    .limit(20)
)